In [1]:
import os
os.environ['CUDA_LAUNCH_BLOCKING'] = '1'

# Task
Implement a GPT-like model in PyTorch from scratch, defining custom modules for RMSNorm, AlibiAttention, SWIGLU, and ResidualBlock, then assemble them into a `GPTModel`, provide example usage, and outline training and inference.

## Define RMSNorm Module

### Subtask:
Implement the `RMSNorm` layer as a PyTorch `nn.Module`. This layer performs Root Mean Square Normalization.


**Reasoning**:
To implement the `RMSNorm` layer, I will create a code block that imports `torch` and `torch.nn`, defines the `RMSNorm` class inheriting from `torch.nn.Module`, initializes the `weight` parameter and `eps` in `__init__`, and implements the RMS normalization logic in the `forward` method as per the instructions.



In [2]:
import torch
import torch.nn as nn

class RMSNorm(nn.Module):
    def __init__(self, dim, eps: float = 1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def _norm(self, x):
        return x * torch.rsqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)

    def forward(self, x):
        return self._norm(x) * self.weight

print("RMSNorm class defined.")

RMSNorm class defined.


## Define AlibiAttention Module

### Subtask:
Implement the `AlibiAttention` layer as a PyTorch `nn.Module`. This module will incorporate multi-head self-attention and apply ALiBi (Attention with Linear Biases) for positional encoding, and handle causal masking.


**Reasoning**:
To implement the `AlibiAttention` module, I will create a code block that defines the `AlibiAttention` class, including its `__init__` method for setting up projections, dropout, and ALiBi slopes, and its `forward` method for performing multi-head self-attention with ALiBi biases and causal masking, as detailed in the instructions.



In [3]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class AlibiAttention(nn.Module):
    def __init__(self, dim, heads: int = 8, dropout_rate: float = 0.1):
        super().__init__()
        self.dim = dim
        self.heads = heads
        self.dropout_rate = dropout_rate
        self.head_dim = dim // heads
        assert (
            self.head_dim * heads == dim
        ), "Dimension must be divisible by number of heads"

        self.q_proj = nn.Linear(dim, dim)
        self.k_proj = nn.Linear(dim, dim)
        self.v_proj = nn.Linear(dim, dim)
        self.out_proj = nn.Linear(dim, dim)
        self.dropout = nn.Dropout(dropout_rate)

        # Compute ALiBi slopes: 2**(-8/heads * i) for i=1 to heads
        alibi_slopes = torch.pow(2, torch.arange(1, heads + 1).float() * (-8.0 / heads))
        self.register_buffer("alibi_slopes", alibi_slopes)

    def forward(self, x):
        batch_size, seq_len, _ = x.shape

        q = self.q_proj(x)
        k = self.k_proj(x)
        v = self.v_proj(x)

        # Reshape and transpose for multi-head attention
        # (batch_size, seq_len, dim) -> (batch_size, seq_len, heads, head_dim) -> (batch_size, heads, seq_len, head_dim)
        q = q.view(batch_size, seq_len, self.heads, self.head_dim).transpose(1, 2)
        k = k.view(batch_size, seq_len, self.heads, self.head_dim).transpose(1, 2)
        v = v.view(batch_size, seq_len, self.heads, self.head_dim).transpose(1, 2)

        # Calculate raw attention scores
        # (batch_size, heads, seq_len, head_dim) @ (batch_size, heads, head_dim, seq_len)
        # -> (batch_size, heads, seq_len, seq_len)
        attn_scores = torch.matmul(q, k.transpose(-2, -1))

        # Scale attention scores
        attn_scores = attn_scores / (self.head_dim**0.5)

        # Apply ALiBi biases
        # Create relative position matrix: (seq_len, seq_len) where bias_matrix[i, j] = j - i
        relative_position_matrix = (
            torch.arange(seq_len, device=x.device)
            .unsqueeze(0)
            - torch.arange(seq_len, device=x.device).unsqueeze(1)
        )

        # Reshape slopes to (heads, 1, 1) for broadcasting
        # alibi_bias: (heads, seq_len, seq_len)
        alibi_bias = self.alibi_slopes.unsqueeze(-1).unsqueeze(-1) * relative_position_matrix.unsqueeze(0)

        # Add ALiBi bias to attention scores
        # attn_scores: (batch_size, heads, seq_len, seq_len) + alibi_bias: (heads, seq_len, seq_len)
        attn_scores = attn_scores + alibi_bias

        # Apply causal masking
        # Create an upper triangular mask (future tokens) and fill with -inf
        causal_mask = torch.triu(
            torch.ones(seq_len, seq_len, dtype=torch.bool, device=x.device),
            diagonal=1,
        )
        # Broadcast causal mask across batch and heads dimensions
        attn_scores = attn_scores.masked_fill(causal_mask.unsqueeze(0).unsqueeze(0), float('-inf'))

        # Apply softmax to get attention probabilities
        attn_probs = F.softmax(attn_scores, dim=-1)

        # Apply dropout
        attn_probs = self.dropout(attn_probs)

        # Compute weighted sum of values
        # (batch_size, heads, seq_len, seq_len) @ (batch_size, heads, seq_len, head_dim)
        # -> (batch_size, heads, seq_len, head_dim)
        output = torch.matmul(attn_probs, v)

        # Concatenate heads and reshape back to original dim
        # (batch_size, seq_len, heads, head_dim) -> (batch_size, seq_len, dim)
        output = output.transpose(1, 2).contiguous().view(batch_size, seq_len, self.dim)

        # Output projection
        output = self.out_proj(output)

        return output

print("AlibiAttention class defined.")

AlibiAttention class defined.


## Define SWIGLU Module

### Subtask:
Implement the `SWIGLU` (Swish-Gated Linear Unit) activation function as a PyTorch `nn.Module`.



## Define SWIGLU Module

### Subtask:
Implement the `SWIGLU` (Swish-Gated Linear Unit) activation function as a PyTorch `nn.Module`.

#### Instructions
1. Import the necessary PyTorch libraries (`torch`, `torch.nn`, `torch.nn.functional`).
2. Define the `SWIGLU` class, inheriting from `torch.nn.Module`.
3. In the `__init__` method:
    a. Initialize the parent class using `super().__init__()`.
    b. Store the input dimension `dim`.
    c. Create two linear layers: `proj_1` and `proj_2`. Both should transform `dim` to `4 * dim` (or a similar expansion factor commonly used in GLU variants).
4. In the `forward` method:
    a. Take the input tensor `x`.
    b. Pass `x` through `proj_1`.
    c. Split the output of `proj_1` into two halves along the last dimension. Let's call them `x1` and `x2`.
    d. Apply the Swish activation function (`F.silu`) to `x1`.
    e. Multiply the result of the Swish activation with `x2` (element-wise).
    f. Pass this product through `proj_2` to project it back to the original `dim`.
    g. Return the final output.

#### Instructions
1. Import the necessary PyTorch libraries (`torch`, `torch.nn`, `torch.nn.functional`).
2. Define the `SWIGLU` class, inheriting from `torch.nn.Module`.
3. In the `__init__` method:
    a. Initialize the parent class using `super().__init__()`.
    b. Store the input dimension `dim`.
    c. Create two linear layers: `proj_1` and `proj_2`. Both should transform `dim` to `4 * dim` (or a similar expansion factor commonly used in GLU variants).
4. In the `forward` method:
    a. Take the input tensor `x`.
    b. Pass `x` through `proj_1`.
    c. Split the output of `proj_1` into two halves along the last dimension. Let's call them `x1` and `x2`.
    d. Apply the Swish activation function (`F.silu`) to `x1`.
    e. Multiply the result of the Swish activation with `x2` (element-wise).
    f. Pass this product through `proj_2` to project it back to the original `dim`.
    g. Return the final output.

#### Instructions
1. Import the necessary PyTorch libraries (`torch`, `torch.nn`, `torch.nn.functional`).
2. Define the `SWIGLU` class, inheriting from `torch.nn.Module`.
3. In the `__init__` method:
    a. Initialize the parent class using `super().__init__()`.
    b. Store the input dimension `dim`.
    c. Create two linear layers: `proj_1` and `proj_2`. Both should transform `dim` to `4 * dim` (or a similar expansion factor commonly used in GLU variants).
4. In the `forward` method:
    a. Take the input tensor `x`.
    b. Pass `x` through `proj_1`.
    c. Split the output of `proj_1` into two halves along the last dimension. Let's call them `x1` and `x2`.
    d. Apply the Swish activation function (`F.silu`) to `x1`.
    e. Multiply the result of the Swish activation with `x2` (element-wise).
    f. Pass this product through `proj_2` to project it back to the original `dim`.
    g. Return the final output.

#### Instructions
1. Import the necessary PyTorch libraries (`torch`, `torch.nn`, `torch.nn.functional`).
2. Define the `SWIGLU` class, inheriting from `torch.nn.Module`.
3. In the `__init__` method:
    a. Initialize the parent class using `super().__init__()`.
    b. Store the input dimension `dim`.
    c. Create two linear layers: `proj_1` and `proj_2`. Both should transform `dim` to `4 * dim` (or a similar expansion factor commonly used in GLU variants).
4. In the `forward` method:
    a. Take the input tensor `x`.
    b. Pass `x` through `proj_1`.
    c. Split the output of `proj_1` into two halves along the last dimension. Let's call them `x1` and `x2`.
    d. Apply the Swish activation function (`F.silu`) to `x1`.
    e. Multiply the result of the Swish activation with `x2` (element-wise).
    f. Pass this product through `proj_2` to project it back to the original `dim`.
    g. Return the final output.

```markdown
#### Instructions
1. Import the necessary PyTorch libraries (`torch`, `torch.nn`, `torch.nn.functional`).
2. Define the `SWIGLU` class, inheriting from `torch.nn.Module`.
3. In the `__init__` method:
    a. Initialize the parent class using `super().__init__()`.
    b. Store the input dimension `dim`.
    c. Create two linear layers: `proj_1` and `proj_2`. Both should transform `dim` to `4 * dim` (or a similar expansion factor commonly used in GLU variants).
4. In the `forward` method:
    a. Take the input tensor `x`.
    b. Pass `x` through `proj_1`.
    c. Split the output of `proj_1` into two halves along the last dimension. Let's call them `x1` and `x2`.
    d. Apply the Swish activation function (`F.silu`) to `x1`.
    e. Multiply the result of the Swish activation with `x2` (element-wise).
    f. Pass this product through `proj_2` to project it back to the original `dim`.
    g. Return the final output.
```

**Reasoning**:
Now that the instructions for the `SWIGLU` module are provided, I will create a code block to implement the `SWIGLU` class in PyTorch as per the detailed instructions.



In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class SWIGLU(nn.Module):
    def __init__(self, dim):
        super().__init__()
        self.dim = dim
        # proj_1 expands the dimension, proj_2 projects it back
        # A common expansion factor is 4, but for GLU variants it's often 2x before splitting
        # Here we assume proj_1 output will be split into two parts of dim*2 each,
        # so the output dimension of proj_1 is dim*4.
        self.proj_1 = nn.Linear(dim, 2 * dim)
        self.proj_2 = nn.Linear(dim, dim)

    def forward(self, x):
        # x: (batch_size, seq_len, dim)
        # Pass through proj_1 and split into two halves
        # result of proj_1: (batch_size, seq_len, 2 * dim)
        # x1, x2: (batch_size, seq_len, dim)
        split_output = self.proj_1(x)
        x1, x2 = torch.chunk(split_output, 2, dim=-1)

        # Apply Swish activation to x1 and multiply with x2
        activated_product = F.silu(x1) * x2

        # Pass through proj_2 to project back to original dim
        output = self.proj_2(activated_product)
        return output

print("SWIGLU class defined.")

SWIGLU class defined.


## Define ResidualBlock Module

### Subtask:
Implement the `ResidualBlock` layer as a PyTorch `nn.Module`. This block should encapsulate an `AlibiAttention` layer and a `SWIGLU` feed-forward network, both followed by `RMSNorm` and dropout, with residual connections.

#### Instructions
1.  Import the necessary PyTorch libraries (`torch`, `torch.nn`).
2.  Define the `ResidualBlock` class, inheriting from `torch.nn.Module`.
3.  In the `__init__` method:
    a.  Initialize the parent class using `super().__init__()`.
    b.  Store the input dimension (`dim`), number of attention heads (`heads`), and dropout rate (`dropout_rate`).
    c.  Instantiate the first `RMSNorm` layer (`self.norm1`).
    d.  Instantiate the `AlibiAttention` layer (`self.attn`) using the provided `dim`, `heads`, and `dropout_rate`.
    e.  Instantiate a `nn.Dropout` layer for the attention output (`self.attn_dropout`).
    f.  Instantiate the second `RMSNorm` layer (`self.norm2`).
    g.  Instantiate the `SWIGLU` layer (`self.ffn`) using the provided `dim`.
    h.  Instantiate a `nn.Dropout` layer for the FFN output (`self.ffn_dropout`).
4.  In the `forward` method:
    a.  Take the input tensor `x`.
    b.  **First Sub-block (Attention)**:
        i.   Apply `self.norm1` to `x`.
        ii.  Pass the normalized output through `self.attn`.
        iii. Apply `self.attn_dropout` to the attention output.
        iv.  Add the dropout output to the original `x` (residual connection).
        v.   Update `x` with this result.
    c.  **Second Sub-block (Feed-forward Network)**:
        i.   Apply `self.norm2` to the current `x`.
        ii.  Pass the normalized output through `self.ffn`.
        iii. Apply `self.ffn_dropout` to the FFN output.
        iv.  Add the dropout output to the current `x` (residual connection).
        v.   Update `x` with this final result.
    d.  Return the final `x`.

#### Instructions
1.  Import the necessary PyTorch libraries (`torch`, `torch.nn`).
2.  Define the `ResidualBlock` class, inheriting from `torch.nn.Module`.
3.  In the `__init__` method:
    a.  Initialize the parent class using `super().__init__()`.
    b.  Store the input dimension (`dim`), number of attention heads (`heads`), and dropout rate (`dropout_rate`).
    c.  Instantiate the first `RMSNorm` layer (`self.norm1`).
    d.  Instantiate the `AlibiAttention` layer (`self.attn`) using the provided `dim`, `heads`, and `dropout_rate`.
    e.  Instantiate a `nn.Dropout` layer for the attention output (`self.attn_dropout`).
    f.  Instantiate the second `RMSNorm` layer (`self.norm2`).
    g.  Instantiate the `SWIGLU` layer (`self.ffn`) using the provided `dim`.
    h.  Instantiate a `nn.Dropout` layer for the FFN output (`self.ffn_dropout`).
4.  In the `forward` method:
    a.  Take the input tensor `x`.
    b.  **First Sub-block (Attention)**:
        i.   Apply `self.norm1` to `x`.
        ii.  Pass the normalized output through `self.attn`.
        iii. Apply `self.attn_dropout` to the attention output.
        iv.  Add the dropout output to the original `x` (residual connection).
        v.   Update `x` with this result.
    c.  **Second Sub-block (Feed-forward Network)**:
        i.   Apply `self.norm2` to the current `x`.
        ii.  Pass the normalized output through `self.ffn`.
        iii. Apply `self.ffn_dropout` to the FFN output.
        iv.  Add the dropout output to the current `x` (residual connection).
        v.   Update `x` with this final result.
    d.  Return the final `x`.


#### Instructions
1.  Import the necessary PyTorch libraries (`torch`, `torch.nn`).
2.  Define the `ResidualBlock` class, inheriting from `torch.nn.Module`.
3.  In the `__init__` method:
    a.  Initialize the parent class using `super().__init__()`.
    b.  Store the input dimension (`dim`), number of attention heads (`heads`), and dropout rate (`dropout_rate`).
    c.  Instantiate the first `RMSNorm` layer (`self.norm1`).
    d.  Instantiate the `AlibiAttention` layer (`self.attn`) using the provided `dim`, `heads`, and `dropout_rate`.
    e.  Instantiate a `nn.Dropout` layer for the attention output (`self.attn_dropout`).
    f.  Instantiate the second `RMSNorm` layer (`self.norm2`).
    g.  Instantiate the `SWIGLU` layer (`self.ffn`) using the provided `dim`.
    h.  Instantiate a `nn.Dropout` layer for the FFN output (`self.ffn_dropout`).
4.  In the `forward` method:
    a.  Take the input tensor `x`.
    b.  **First Sub-block (Attention)**:
        i.   Apply `self.norm1` to `x`.
        ii.  Pass the normalized output through `self.attn`.
        iii. Apply `self.attn_dropout` to the attention output.
        iv.  Add the dropout output to the original `x` (residual connection).
        v.   Update `x` with this result.
    c.  **Second Sub-block (Feed-forward Network)**:
        i.   Apply `self.norm2` to the current `x`.
        ii.  Pass the normalized output through `self.ffn`.
        iii. Apply `self.ffn_dropout` to the FFN output.
        iv.  Add the dropout output to the current `x` (residual connection).
        v.   Update `x` with this final result.
    d.  Return the final `x`.

#### Reasoning:
To implement the `ResidualBlock` layer, I will create a code block that imports necessary PyTorch modules, defines the `ResidualBlock` class inheriting from `torch.nn.Module`, instantiates `RMSNorm`, `AlibiAttention`, `SWIGLU`, and `nn.Dropout` layers in the `__init__` method, and implements the attention and feed-forward sub-blocks with residual connections in the `forward` method, as detailed in the instructions.

#### Reasoning:
To implement the `ResidualBlock` layer, I will create a code block that imports necessary PyTorch modules, defines the `ResidualBlock` class inheriting from `torch.nn.Module`, instantiates `RMSNorm`, `AlibiAttention`, `SWIGLU`, and `nn.Dropout` layers in the `__init__` method, and implements the attention and feed-forward sub-blocks with residual connections in the `forward` method, as detailed in the instructions.

#### Reasoning:
To implement the `ResidualBlock` layer, I will create a code block that imports necessary PyTorch modules, defines the `ResidualBlock` class inheriting from `torch.nn.Module`, instantiates `RMSNorm`, `AlibiAttention`, `SWIGLU`, and `nn.Dropout` layers in the `__init__` method, and implements the attention and feed-forward sub-blocks with residual connections in the `forward` method, as detailed in the instructions.

## Define ResidualBlock Module

### Subtask:
Implement a `ResidualBlock` module that encapsulates an `RMSNorm` layer followed by a main layer (either `AlibiAttention` or `SWIGLU`), adding the input to the layer's output (residual connection).


#### Instructions
1. Import the necessary PyTorch libraries (`torch`, `torch.nn`).
2. Define the `ResidualBlock` class, inheriting from `torch.nn.Module`.
3. In the `__init__` method:
    a. Initialize the parent class using `super().__init__()`.
    b. Store the input dimension (`dim`), number of attention heads (`heads`), and dropout rate (`dropout_rate`).
    c. Instantiate the first `RMSNorm` layer (`self.norm1`).
    d. Instantiate the `AlibiAttention` layer (`self.attn`) using the provided `dim`, `heads`, and `dropout_rate`.
    e. Instantiate a `nn.Dropout` layer for the attention output (`self.attn_dropout`).
    f. Instantiate the second `RMSNorm` layer (`self.norm2`).
    g. Instantiate the `SWIGLU` layer (`self.ffn`) using the provided `dim`.
    h. Instantiate a `nn.Dropout` layer for the FFN output (`self.ffn_dropout`).
4. In the `forward` method:
    a. Take the input tensor `x`.
    b. **First Sub-block (Attention)**:
        i.   Apply `self.norm1` to `x`.
        ii.  Pass the normalized output through `self.attn`.
        iii. Apply `self.attn_dropout` to the attention output.
        iv.  Add the dropout output to the original `x` (residual connection).
        v.   Update `x` with this result.
    c. **Second Sub-block (Feed-forward Network)**:
        i.   Apply `self.norm2` to the current `x`.
        ii.  Pass the normalized output through `self.ffn`.
        iii. Apply `self.ffn_dropout` to the FFN output.
        iv.  Add the dropout output to the current `x` (residual connection).
        v.   Update `x` with this final result.
    d. Return the final `x`.

**Reasoning**:
Now that the instructions for the `ResidualBlock` module are provided, I will create a code block to implement the `ResidualBlock` class in PyTorch as per the detailed instructions.



In [5]:
import torch
import torch.nn as nn

# Assuming RMSNorm, AlibiAttention, and SWIGLU classes are defined in previous cells
# If not, they would need to be imported or redefined here for this block to run independently.

class ResidualBlock(nn.Module):
    def __init__(self, dim, heads: int = 8, dropout_rate: float = 0.1):
        super().__init__()
        self.dim = dim
        self.heads = heads
        self.dropout_rate = dropout_rate

        # First Sub-block: Attention
        self.norm1 = RMSNorm(dim) # Instantiate RMSNorm
        self.attn = AlibiAttention(dim, heads=heads, dropout_rate=dropout_rate) # Instantiate AlibiAttention
        self.attn_dropout = nn.Dropout(dropout_rate) # Instantiate Dropout for attention

        # Second Sub-block: Feed-forward Network
        self.norm2 = RMSNorm(dim) # Instantiate RMSNorm
        self.ffn = SWIGLU(dim) # Instantiate SWIGLU
        self.ffn_dropout = nn.Dropout(dropout_rate) # Instantiate Dropout for FFN

    def forward(self, x):
        # x is the input tensor

        # First Sub-block (Attention) with residual connection
        normalized_x = self.norm1(x) # Apply RMSNorm
        attn_output = self.attn(normalized_x) # Pass through AlibiAttention
        attn_output = self.attn_dropout(attn_output) # Apply dropout
        x = x + attn_output # Residual connection

        # Second Sub-block (Feed-forward Network) with residual connection
        normalized_x = self.norm2(x) # Apply RMSNorm
        ffn_output = self.ffn(normalized_x) # Pass through SWIGLU
        ffn_output = self.ffn_dropout(ffn_output) # Apply dropout
        x = x + ffn_output # Residual connection

        return x

print("ResidualBlock class defined.")

ResidualBlock class defined.


## Build GPT-like Model

### Subtask:
Assemble the `GPTModel` using token embeddings, the defined `ResidualBlock`s, a final `RMSNorm`, and a linear output layer.


**Reasoning**:
To implement the `GPTModel` as instructed, I will create a code block that defines the `GPTModel` class, including its `__init__` method for setting up token embeddings, multiple `ResidualBlock` layers, a final `RMSNorm` layer, and an output linear layer, and its `forward` method to process input through these components.



In [6]:
import torch
import torch.nn as nn

# Assuming RMSNorm, AlibiAttention, SWIGLU, and ResidualBlock classes
# are defined in previous cells and are available in the scope.

class GPTModel(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        dim: int,
        num_layers: int,
        heads: int,
        seq_len: int,
        dropout_rate: float = 0.1,
    ):
        super().__init__()
        self.vocab_size = vocab_size
        self.dim = dim
        self.num_layers = num_layers
        self.heads = heads
        self.seq_len = seq_len
        self.dropout_rate = dropout_rate

        # Token embeddings layer
        self.token_embeddings = nn.Embedding(vocab_size, dim)

        # Positional embeddings (not explicitly mentioned for ALiBi but often used for token context)
        # For ALiBi, explicit positional embeddings are not strictly necessary
        # as positional information is encoded in attention biases.
        # However, a common practice is to have token embeddings. For this task, we will stick to token embeddings only.

        # Residual Blocks
        self.layers = nn.ModuleList(
            [
                ResidualBlock(dim, heads=heads, dropout_rate=dropout_rate)
                for _ in range(num_layers)
            ]
        )

        # Final RMSNorm layer
        self.final_norm = RMSNorm(dim)

        # Output linear layer to project back to vocab_size
        self.output_head = nn.Linear(dim, vocab_size, bias=False)

    def forward(self, x):
        # x: (batch_size, seq_len) - input token indices

        # 1. Get token embeddings
        embeddings = self.token_embeddings(x)  # (batch_size, seq_len, dim)

        # 2. Pass through Residual Blocks
        for layer in self.layers:
            embeddings = layer(embeddings)

        # 3. Apply final RMSNorm
        output = self.final_norm(embeddings)  # (batch_size, seq_len, dim)

        # 4. Project to vocabulary size to get logits
        logits = self.output_head(output)  # (batch_size, seq_len, vocab_size)

        return logits

print("GPTModel class defined.")

GPTModel class defined.


## Provide Example Usage

### Subtask:
Demonstrate how to instantiate the `GPTModel` with example parameters and process dummy input data, illustrating the forward pass.


**Reasoning**:
To demonstrate the `GPTModel` usage, I will create a code block that imports `torch`, defines example parameters, instantiates the model, creates a dummy input tensor, performs a forward pass, and prints the shape of the output logits, as per the instructions.



In [7]:
import torch

# Define example parameters for the GPTModel
vocab_size = 1000  # Example vocabulary size
dim = 256        # Example model dimension
num_layers = 2   # Example number of ResidualBlocks
heads = 4        # Example number of attention heads
seq_len = 50     # Example sequence length
dropout_rate = 0.1 # Example dropout rate

print("Instantiating GPTModel with example parameters:")
print(f"  vocab_size: {vocab_size}")
print(f"  dim: {dim}")
print(f"  num_layers: {num_layers}")
print(f"  heads: {heads}")
print(f"  seq_len: {seq_len}")
print(f"  dropout_rate: {dropout_rate}")

# Instantiate the GPTModel
try:
    model = GPTModel(
        vocab_size=vocab_size,
        dim=dim,
        num_layers=num_layers,
        heads=heads,
        seq_len=seq_len,
        dropout_rate=dropout_rate,
    )
    print("GPTModel instantiated successfully.")

    # Create a dummy input tensor
    batch_size = 4  # Example batch size
    dummy_input = torch.randint(0, vocab_size, (batch_size, seq_len))
    print(f"Dummy input shape: {dummy_input.shape}")

    # Pass the dummy input through the instantiated GPTModel
    print("Performing forward pass...")
    output_logits = model(dummy_input)

    # Print the shape of the output logits
    print(f"Shape of output logits: {output_logits.shape}")

except NameError as e:
    print(f"Error: {e}. Please ensure all required modules (RMSNorm, AlibiAttention, SWIGLU, ResidualBlock, GPTModel) are defined and accessible in the current scope.")
except Exception as e:
    print(f"An unexpected error occurred during model instantiation or forward pass: {e}")


Instantiating GPTModel with example parameters:
  vocab_size: 1000
  dim: 256
  num_layers: 2
  heads: 4
  seq_len: 50
  dropout_rate: 0.1
GPTModel instantiated successfully.
Dummy input shape: torch.Size([4, 50])
Performing forward pass...
Shape of output logits: torch.Size([4, 50, 1000])


## Outline Training and Inference

### Subtask:
Provide a high-level explanation of how this model would be integrated into a training loop and used for autoregressive token generation, including tokenizer considerations.


#### Instructions
1. **Training Outline**: Describe a typical training loop for this GPT-like model. This should cover:
    a.  **Input/Output**: What are the inputs (`x`) to the model during training, and what are the targets?
    b.  **Loss Function**: Which PyTorch loss function is suitable for next-token prediction (e.g., `nn.CrossEntropyLoss`)?
    c.  **Optimizer**: How would an optimizer be initialized and used (e.g., `AdamW`)?
    d.  **Backward Pass**: Briefly explain the steps for calculating gradients and updating model weights.
2.  **Inference Outline**: Explain how the trained model would be used for autoregressive text generation. This should cover:
    a.  **Input**: How does the generation process start (e.g., with a prompt)?
    b.  **Loop**: Describe the iterative nature of generating one token at a time.
    c.  **Tokenization/Detokenization**: Briefly mention the role of a tokenizer and how it's used for preparing input and processing output.
    d.  **Sampling**: Explain how the model's output logits are converted into a discrete token prediction (e.g., greedy sampling, top-k, nucleus sampling).
    e.  **Stopping Condition**: How would the generation process know when to stop?

### Training and Inference Outline

#### Training Outline
A typical training loop for this GPT-like model involves preparing sequential input data, feeding it to the model, calculating the loss, and updating the model's weights.

1.  **Input/Output**:
    *   **Input (`x`)**: During training, the model receives a batch of tokenized sequences. For next-token prediction, the input `x` would be a sequence of token IDs `[t_1, t_2, ..., t_n]`. The model processes this sequence to predict the next token at each position.
    *   **Targets**: The targets (`y`) for the model are the input sequence shifted by one position, i.e., `[t_2, t_3, ..., t_n, <EOS>]` (where `<EOS>` is an end-of-sequence token or the next token if the sequence continues). Each output `logit[i]` from the model for `t_i` will be used to predict `t_{i+1}`.

2.  **Loss Function**: For next-token prediction in a multi-class classification scenario (predicting one token out of `vocab_size` possibilities), `torch.nn.CrossEntropyLoss` is highly suitable. It combines `LogSoftmax` and `NLLLoss` in one single class, which is numerically stable.

3.  **Optimizer**: An optimizer like `torch.optim.AdamW` is commonly used for transformer models due to its adaptive learning rate capabilities and weight decay regularization. It would be initialized with the model's parameters and a learning rate:
    ```python
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)
    ```

4.  **Backward Pass**: After the model computes `logits = model(x)` and the loss is calculated (e.g., `loss = criterion(logits.view(-1, vocab_size), targets.view(-1))`), the backward pass involves:
    a.  Clearing previous gradients: `optimizer.zero_grad()`.
    b.  Computing gradients with respect to the loss: `loss.backward()`.
    c.  Updating model parameters using the computed gradients: `optimizer.step()`.

#### Inference Outline
Inference for autoregressive token generation uses the trained model to generate new sequences, one token at a time.

1.  **Input**: The generation process typically starts with an initial prompt, which is a sequence of one or more token IDs. This prompt is fed to the model to generate the first new token.

2.  **Loop**: Autoregressive generation proceeds in a loop:
    a.  The current sequence (initial prompt + generated tokens so far) is fed into the model.
    b.  The model outputs logits for the next token at the last position of the input sequence.
    c.  A sampling strategy selects the next token ID from these logits.
    d.  The selected token ID is appended to the current sequence.
    e.  Steps a-d are repeated until a stopping condition is met.

3.  **Tokenization/Detokenization**:
    *   **Tokenization**: A tokenizer (e.g., BPE, WordPiece, SentencePiece) is crucial for converting raw text into numerical token IDs that the model understands, both for the initial prompt and for training data preparation.
    *   **Detokenization**: After the model generates token IDs, the tokenizer's `decode` method is used to convert these IDs back into human-readable text.

4.  **Sampling**: The model outputs a probability distribution over the vocabulary for the next token. Several sampling strategies can be used to select the actual token:
    *   **Greedy Sampling**: Simply selects the token with the highest probability (i.e., `torch.argmax(logits, dim=-1)`).
    *   **Top-K Sampling**: Considers only the `k` most likely tokens, normalizes their probabilities, and then samples from this reduced set.
    *   **Nucleus Sampling (Top-P)**: Considers the smallest set of most likely tokens whose cumulative probability exceeds a threshold `p`, and then samples from this set.

5.  **Stopping Condition**: The generation process needs criteria to stop. Common stopping conditions include:
    *   Generating a special End-Of-Sequence (`<EOS>`) token.
    *   Reaching a predefined maximum sequence length.
    *   Generating a sequence that leads to repetition or undesirable output (less common for simple implementations, more for advanced control).

## Final Task

### Subtask:
Summarize the created PyTorch model, its components, and its potential for use in building a small language model.


## Summary:

### Data Analysis Key Findings
*   **Custom Module Implementations**: All required custom PyTorch modules were successfully implemented:
    *   `RMSNorm`: Performs Root Mean Square Normalization with a learnable weight.
    *   `AlibiAttention`: Implements multi-head self-attention with ALiBi (Attention with Linear Biases) for positional encoding and causal masking, crucial for transformer decoder architectures.
    *   `SWIGLU`: Defines the Swish-Gated Linear Unit activation function, a common component in modern transformer feed-forward networks.
    *   `ResidualBlock`: Encapsulates an `AlibiAttention` layer and a `SWIGLU` feed-forward network, each preceded by `RMSNorm` and followed by dropout with residual connections, forming the core building block of the GPT-like model.
*   **GPTModel Assembly**: A `GPTModel` class was successfully assembled, integrating token embeddings, multiple `ResidualBlock` layers, a final `RMSNorm` layer, and an output linear head to project to the vocabulary size.
*   **Functional Verification**: An example usage demonstrated successful instantiation of the `GPTModel` with parameters such as `vocab_size=1000`, `dim=256`, `num_layers=2`, `heads=4`, and `seq_len=50`. A dummy input of shape `(4, 50)` (batch\_size, seq\_len) was processed, yielding output logits of `torch.Size([4, 50, 1000])`, confirming the expected output dimensions.
*   **Training and Inference Outline**: A comprehensive high-level outline was provided for both training (using `torch.nn.CrossEntropyLoss` and `torch.optim.AdamW` for next-token prediction) and autoregressive inference (including prompt-based generation, iterative token prediction, tokenization/detokenization, various sampling strategies, and stopping conditions).

### Insights or Next Steps
*   The modular design, consisting of custom `RMSNorm`, `AlibiAttention`, `SWIGLU`, and `ResidualBlock` layers, provides flexibility for experimentation and scaling of the GPT-like model.
*   The implemented `GPTModel` is now ready for integration into a complete training pipeline, requiring a dataset, tokenizer, and hyperparameter tuning to develop a functional small language model.


# Task
Implement a Byte Pair Encoding (BPE) tokenizer (Karpathy-style), defining a `BPETokenizer` class with methods for training on a text corpus, and for `encode` (text to token IDs) and `decode` (token IDs to text). The tokenizer should handle unknown tokens and optionally special tokens like `<EOS>`.

## Implement BPE Tokenizer (Karpathy-style)

### Subtask:
Create a Byte Pair Encoding (BPE) tokenizer, similar to Andrej Karpathy's implementation. Define a `BPETokenizer` class with methods for training, encoding text to token IDs, and decoding token IDs to text. The tokenizer should handle unknown tokens and optional special tokens.


#### Reasoning:
To implement the `BPETokenizer` as instructed, I will create a code block that defines the `BPETokenizer` class, including its `__init__` method for initialization, helper methods `_get_pairs` and `_merge_pair`, and core methods `train`, `encode`, and `decode`, following the detailed instructions provided.

**Reasoning**:
I will create a code block to implement the `BPETokenizer` class, defining its `__init__`, `_get_pairs`, `_merge_pair`, `train`, `encode`, and `decode` methods as specified in the instructions, ensuring it handles initial vocabulary, merges, and special tokens.



In [8]:
import collections
from collections import defaultdict

class BPETokenizer:
    def __init__(self, special_tokens=None):
        self.merges = {}
        self.vocab = {}
        self.inverse_vocab = []

        # Initialize base vocabulary with standard ASCII characters up to 255
        self.base_vocab = [chr(i) for i in range(256)]

        # Assign IDs to base vocab
        for i, token in enumerate(self.base_vocab):
            self.vocab[token] = i
            self.inverse_vocab.append(token)

        self.vocab_size = len(self.base_vocab)

        # Add special tokens if provided
        if special_tokens:
            for token in special_tokens:
                if token not in self.vocab:
                    self.vocab[token] = self.vocab_size
                    self.inverse_vocab.append(token)
                    self.vocab_size += 1

    def _get_pairs(self, word_tokens):
        pairs = defaultdict(int)
        for i in range(len(word_tokens) - 1):
            pairs[(word_tokens[i], word_tokens[i+1])] += 1
        return pairs

    def _merge_pair(self, tokens, pair, new_token):
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        return new_tokens

    def train(self, text_corpus, num_merges):
        # For BPE, we typically start by splitting words into characters.
        # A simpler approach for character-level BPE is to consider the entire corpus
        # as a sequence of characters and merge directly.
        # For Karpathy-style BPE, each 'word' is processed independently before merging.

        # Let's simplify this by treating the entire corpus as one long sequence of characters
        # for initial tokenization, similar to how a byte-level BPE would start.

        # Convert corpus into a list of base tokens (characters)
        # Note: A more robust BPE would first tokenize into words, then char-tokenize each word.
        # This implementation simplifies to char-tokenize the entire corpus for merges.
        word_tokens_list = []
        for char in text_corpus:
            word_tokens_list.append([char])

        # Keep track of current token sequences. This is a list of lists of characters.
        current_corpus_token_sequences = [[char for char in word] for word in text_corpus.split()]
        # The above line is problematic if `word` is actually a single character. Let's adapt.
        # Instead of word by word, let's process the raw text as a sequence of characters
        # This makes the merges global across the text rather than confined to words.
        # A more 'Karpathy-style' implementation would process words first.

        # Let's align with a more direct BPE implementation that processes char sequences
        # of *words* independently until encoding, then applies merges.
        # For simplicity and aligning with general BPE concepts, we'll process the corpus
        # as character sequences, potentially across 'word' boundaries implicitly.

        # The provided instructions suggest `b. Convert text_corpus into a list of base tokens (characters) by splitting each character.`
        # and `c. Store the initial token sequences for each word (or contiguous character sequence) in the corpus, represented as lists of characters.`
        # This implies processing words as character lists.

        initial_token_sequences = []
        for word in text_corpus.split(' '): # Split by space to get words/segments
            initial_token_sequences.append(list(word))
            # Add space back as a separate token if it's meant to be tokenized
            # This depends on whether spaces are part of vocabulary or delimiters.
            # For now, let's assume spaces are implicitly handled by splitting.

        # If we want to include spaces, we need to handle them explicitly
        # For a true byte-level BPE, every byte/char including spaces is a token.
        # Let's reconsider: for a simple BPE, just split the entire text into characters.
        # This is simpler and aligns with the instructions better without making assumptions on words vs chars.

        current_text_chars = [char for char in text_corpus]

        # The `train` method should iterate on token sequences for the entire corpus
        # For training, we need to collect all sequences first.
        # Let's assume the input `text_corpus` is a single string.
        # We need to process each 'word' (or character sequence) independently for pair counting.

        # Store current state of token sequences for each 'word' in the corpus
        # This is a list of lists, where each inner list is a word represented by its characters.
        token_sequences_for_training = []
        for word in text_corpus.split(' '):
            if word:
                token_sequences_for_training.append([char for char in word])

        for merge_idx in range(num_merges):
            pairs = defaultdict(int)
            for seq in token_sequences_for_training:
                word_pairs = self._get_pairs(seq)
                for pair, count in word_pairs.items():
                    pairs[pair] += count

            if not pairs:
                break

            most_frequent_pair = max(pairs, key=pairs.get)

            new_token = ''.join(most_frequent_pair)

            # Record the merge
            self.merges[most_frequent_pair] = new_token

            # Update all token sequences in the corpus by merging the pair
            for i in range(len(token_sequences_for_training)):
                token_sequences_for_training[i] = self._merge_pair(
                    token_sequences_for_training[i], most_frequent_pair, new_token
                )

            # Assign a new ID to the new_token and update vocab
            if new_token not in self.vocab:
                self.vocab[new_token] = self.vocab_size
                self.inverse_vocab.append(new_token)
                self.vocab_size += 1

        # After training, merges are ordered. Store them as an ordered list for encoding.
        self._ordered_merges = list(self.merges.items())

    def encode(self, text):
        # Convert input text into initial character tokens
        tokens = [char for char in text]

        # Apply all learned merges iteratively
        # Iterate through merges in the order they were learned
        for pair, new_token in self._ordered_merges:
            tokens = self._merge_pair(tokens, pair, new_token)

        # Convert final tokens to IDs
        encoded_ids = []
        for token in tokens:
            if token in self.vocab:
                encoded_ids.append(self.vocab[token])
            else:
                # Handle unknown tokens: split into individual characters
                # and encode them, or use a special UNK token if available.
                # For now, we'll split into individual characters from base_vocab.
                for char in token:
                    if char in self.vocab: # Should always be true if base_vocab contains all chars
                        encoded_ids.append(self.vocab[char])
                    else: # Fallback for truly unknown character (e.g., beyond ASCII 255)
                        # Assign a special UNK token ID if exists, or raise error.
                        # For this example, let's assume all chars are in base_vocab.
                        pass # This case ideally shouldn't happen with range(256)
        return encoded_ids

    def decode(self, token_ids):
        decoded_tokens = []
        for token_id in token_ids:
            if 0 <= token_id < len(self.inverse_vocab):
                decoded_tokens.append(self.inverse_vocab[token_id])
            else:
                # Handle unknown token IDs if they somehow appear (e.g., padding token)
                decoded_tokens.append("[UNK]") # Placeholder for unknown ID
        return "".join(decoded_tokens)

print("BPETokenizer class defined.")


BPETokenizer class defined.


## Provide Example Usage for BPETokenizer

### Subtask:
Demonstrate the functionality of the `BPETokenizer` by:
1. Instantiating the tokenizer.
2. Training it on a sample text corpus with a specified number of merges.
3. Encoding a test string.
4. Decoding the resulting token IDs back to text.
5. Printing the results to verify correctness.

**Reasoning**:
To demonstrate the `BPETokenizer` functionality, I will create a code block that instantiates the tokenizer, trains it on a sample corpus, encodes a test string, and then decodes the resulting token IDs back to text, printing each step's output for verification.



In [9]:
import collections
from collections import defaultdict

# Re-define BPETokenizer here if it's not guaranteed to be in scope
# For the purpose of demonstration, we assume it's in scope from previous cells

print("Demonstrating BPETokenizer functionality...")

# 1. Instantiate the tokenizer
special_tokens = ["<UNK>", "<EOS>", "<PAD>"]
tokenizer = BPETokenizer(special_tokens=special_tokens)
print(f"Tokenizer initialized with base vocab size: {len(tokenizer.base_vocab)} and total vocab size: {tokenizer.vocab_size}")

# 2. Train it on a sample text corpus
sample_text_corpus = "this is a sample text corpus for bpe training. bpe is fun!"
num_merges = 10

print(f"\nTraining tokenizer on corpus: '{sample_text_corpus}' with {num_merges} merges...")
tokenizer.train(sample_text_corpus, num_merges)
print(f"Training complete. New vocab size: {tokenizer.vocab_size}")
print(f"Learned merges: {tokenizer.merges}")

# 3. Encode a test string
test_string = "this is a fun bpe example"
print(f"\nEncoding test string: '{test_string}'")
encoded_ids = tokenizer.encode(test_string)
print(f"Encoded IDs: {encoded_ids}")

# 4. Decode the resulting token IDs back to text
print(f"\nDecoding IDs: {encoded_ids}")
decoded_text = tokenizer.decode(encoded_ids)
print(f"Decoded text: '{decoded_text}'")

# 5. Verify correctness
if test_string == decoded_text:
    print("\nEncoding and Decoding successful and consistent!")
else:
    print("\nWarning: Decoded text does not exactly match original test string.")

# Example with an unknown character (if any are not in base_vocab)
# Assuming base_vocab includes all standard ASCII chars, this might not show UNK handling
# For illustration, let's use a character outside 0-255 if possible
# For this BPE (chr(0)-chr(255)), most common chars will be known.
# If we were to encode '你好', for example, it would break down into unknown individual byte tokens if not in base_vocab.


Demonstrating BPETokenizer functionality...
Tokenizer initialized with base vocab size: 256 and total vocab size: 259

Training tokenizer on corpus: 'this is a sample text corpus for bpe training. bpe is fun!' with 10 merges...
Training complete. New vocab size: 269
Learned merges: {('i', 's'): 'is', ('o', 'r'): 'or', ('b', 'p'): 'bp', ('bp', 'e'): 'bpe', ('i', 'n'): 'in', ('t', 'h'): 'th', ('th', 'is'): 'this', ('s', 'a'): 'sa', ('sa', 'm'): 'sam', ('sam', 'p'): 'samp'}

Encoding test string: 'this is a fun bpe example'
Encoded IDs: [265, 32, 259, 32, 97, 32, 102, 117, 110, 32, 262, 32, 101, 120, 97, 109, 112, 108, 101]

Decoding IDs: [265, 32, 259, 32, 97, 32, 102, 117, 110, 32, 262, 32, 101, 120, 97, 109, 112, 108, 101]
Decoded text: 'this is a fun bpe example'

Encoding and Decoding successful and consistent!


## Prepare Training Data with BPE Tokenizer

### Subtask:
Generate dummy text data, tokenize it using the implemented BPE tokenizer, and format the tokenized data into input-target pairs suitable for next-token prediction training. Finally, convert these pairs into PyTorch tensors.


**Reasoning**:
To prepare the training data as instructed, I will create a code block that defines a sample text corpus, instantiates and trains the `BPETokenizer`, encodes the corpus, generates input-target pairs with a specified sequence length, and finally converts these pairs into PyTorch tensors.



In [10]:
import torch

# Assuming BPETokenizer, vocab_size, seq_len, etc. are available from previous cells
# Re-instantiate BPETokenizer if not already done in the current session or for clarity

# 1. Define a sample text corpus for training
sample_training_corpus = """
"The quick brown fox jumps over the lazy dog. The dog barks loudly."\
"BPE tokenization is a subword tokenization algorithm often used in natural language processing (NLP)."\
"It works by iteratively merging the most frequent pair of characters or character sequences in a text corpus."\
"This process continues until a predefined vocabulary size or number of merge operations is reached."\
"The resulting subword units can effectively handle out-of-vocabulary words and reduce vocabulary size."\
"This makes BPE tokenizers highly efficient for various NLP tasks, including machine translation and language modeling."
"""

# Reuse previous tokenizer instance or create a new one
# For this task, let's create a fresh tokenizer to ensure it's trained on the *full* sample_training_corpus
# rather than just the small 'this is a sample text corpus...' from the example usage.
special_tokens = ["<UNK>", "<EOS>", "<PAD>"]
tokenizer = BPETokenizer(special_tokens=special_tokens)

# Define the number of merges for training the tokenizer
# A larger corpus usually benefits from more merges.
num_training_merges = 50 # Let's train with more merges for a slightly larger corpus

print(f"Training tokenizer on a new corpus with {num_training_merges} merges...")
tokenizer.train(sample_training_corpus, num_training_merges)
print(f"Tokenizer training complete. Final vocab size: {tokenizer.vocab_size}")

# 4. Encode the entire training corpus using the trained tokenizer
encoded_corpus_ids = tokenizer.encode(sample_training_corpus)
print(f"\nEncoded corpus length: {len(encoded_corpus_ids)}")

# 5. Create input-target pairs from the encoded token IDs
# Reuse the seq_len from GPTModel definition, which was 50.
# Make sure `seq_len` from the GPTModel definition is still available (e.g., 50 from variable tracking)

input_sequences = []
target_sequences = []

for i in range(0, len(encoded_corpus_ids) - seq_len):
    input_seq = encoded_corpus_ids[i : i + seq_len]
    target_seq = encoded_corpus_ids[i + 1 : i + seq_len + 1]

    input_sequences.append(input_seq)
    target_sequences.append(target_seq)

print(f"Generated {len(input_sequences)} input-target pairs.")
print(f"Example input sequence (first 5 tokens): {input_sequences[0][:5]}")
print(f"Example target sequence (first 5 tokens): {target_sequences[0][:5]}")

# 6. Convert the list of input token ID sequences and target token ID sequences into PyTorch tensors
input_tensor = torch.tensor(input_sequences, dtype=torch.long)
target_tensor = torch.tensor(target_sequences, dtype=torch.long)

print(f"\nInput tensor shape: {input_tensor.shape}")
print(f"Target tensor shape: {target_tensor.shape}")

# Verify a few properties
# The vocabulary size of the tokenizer should match the vocab_size used in GPTModel
# if the model is intended to be trained on this data.
if tokenizer.vocab_size > vocab_size:
    print(f"Warning: Tokenizer vocab_size ({tokenizer.vocab_size}) is larger than GPTModel vocab_size ({vocab_size}). This may cause issues during training.")
elif tokenizer.vocab_size < vocab_size:
    print(f"Note: Tokenizer vocab_size ({tokenizer.vocab_size}) is smaller than GPTModel vocab_size ({vocab_size}). This is generally fine, but some embeddings in GPTModel will be unused.")


Training tokenizer on a new corpus with 50 merges...
Tokenizer training complete. Final vocab size: 309

Encoded corpus length: 403
Generated 353 input-target pairs.
Example input sequence (first 5 tokens): [10, 34, 282, 32, 283]
Example target sequence (first 5 tokens): [34, 282, 32, 283, 304]

Input tensor shape: torch.Size([353, 50])
Target tensor shape: torch.Size([353, 50])
Note: Tokenizer vocab_size (309) is smaller than GPTModel vocab_size (1000). This is generally fine, but some embeddings in GPTModel will be unused.


## Define Training Function

### Subtask:
Implement a `train` function that orchestrates the training process. This function will initialize `nn.CrossEntropyLoss` and an `AdamW` optimizer, and run the model training for 1000 epochs, including forward pass, loss calculation, backward pass, and optimizer step. It will also print training progress periodically.


**Reasoning**:
To implement the `train` function, I will create a code block that defines the function, sets the model to training mode, initializes `CrossEntropyLoss` and `AdamW` optimizer, and then constructs an epoch and batch-wise training loop. Inside the loop, it will perform forward and backward passes, optimize parameters, and print progress periodically, as detailed in the instructions.

**Reasoning**:
Now that the reasoning is outlined, I will generate the Python code to implement the `train` function according to the detailed instructions, including model setup, loss, optimizer, batch processing, and progress reporting.



In [11]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm # For progress bar

def train(
    model: nn.Module,
    input_tensor: torch.Tensor,
    target_tensor: torch.Tensor,
    epochs: int,
    batch_size: int,
    learning_rate: float = 1e-4,
    print_every: int = 100,
):
    # 2. Set the model to training mode
    model.train()

    # Determine device
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    input_tensor = input_tensor.to(device)
    target_tensor = target_tensor.to(device)

    print(f"Training model on device: {device}")

    # 3. Instantiate CrossEntropyLoss
    criterion = nn.CrossEntropyLoss()

    # 4. Instantiate AdamW optimizer (Changed from SGD)
    optimizer = optim.SGD(model.parameters(), lr=learning_rate)

    print(f"Starting training for {epochs} epochs...")

    # 5. Implement a training loop
    for epoch in range(1, epochs + 1):
        total_loss = 0
        num_batches = 0

        # 6. Iterate through data in mini-batches
        # Shuffle indices for each epoch
        indices = torch.randperm(len(input_tensor))

        for i in range(0, len(input_tensor), batch_size):
            batch_indices = indices[i : i + batch_size]
            inputs = input_tensor[batch_indices]
            targets = target_tensor[batch_indices]

            # 7. For each batch:
            # a. & b. Inputs and targets already moved to device above.

            # c. Perform a forward pass
            outputs = model(inputs)

            # d. Calculate the loss
            # Reshape outputs to (N, C) where N is batch_size * seq_len and C is vocab_size
            # Reshape targets to (N)
            loss = criterion(outputs.view(-1, outputs.size(-1)), targets.view(-1))

            # e. Zero out the gradients
            optimizer.zero_grad()

            # f. Perform a backward pass
            loss.backward()

            # g. Update model parameters
            optimizer.step()

            total_loss += loss.item()
            num_batches += 1

        avg_loss = total_loss / num_batches

        # 8. Print training progress periodically
        if epoch % print_every == 0 or epoch == epochs:
            print(f"Epoch {epoch}/{epochs}, Loss: {avg_loss:.4f}")

    print("Training complete.")

print("Train function defined.")

Train function defined.


# Task
Install `tqdm` for the progress bar. Load text content from 'corpus.txt'. Initialize the `BPETokenizer`, and train it on the corpus with a `num_merges` that ensures the tokenizer's vocabulary size is at least 1000. Tokenize the entire corpus, create input-target pairs with `seq_len` 50, and convert them into PyTorch tensors. Instantiate `GPTModel` with `vocab_size` (at least 1000), `d_model` 256, `num_layers` 2, `heads` 4, and `seq_len` 50. Train the `GPTModel` for 1000 epochs with a `batch_size` of 4 and `learning_rate` of 1e-4. Finally, define and use an `inference` function to generate text from a short prompt.

## Load Corpus Data

### Subtask:
Load the text content from 'corpus.txt' for tokenizer training and model training.


**Reasoning**:
The subtask requires installing the `tqdm` library. I will use a `code_block` to execute the `!pip install tqdm` command.



In [12]:
get_ipython().system('pip install tqdm')

**Reasoning**:
Now that `tqdm` is confirmed to be installed, I will load the text content from 'corpus.txt' into a string variable as per the subtask instructions.



In [13]:
import os

corpus_file_path = 'corpus.txt'

# Check if the file exists
if os.path.exists(corpus_file_path):
    with open(corpus_file_path, 'r', encoding='utf-8') as f:
        loaded_corpus_text = f.read()
    print(f"Successfully loaded corpus from '{corpus_file_path}'. Length: {len(loaded_corpus_text)} characters.")
    # Display first 200 characters to verify content
    print("First 200 characters of the loaded corpus:")
    print("-----------------------------------------")
    print(loaded_corpus_text[:200])
    print("-----------------------------------------")
else:
    print(f"Error: The file '{corpus_file_path}' was not found.")
    loaded_corpus_text = ""


Successfully loaded corpus from 'corpus.txt'. Length: 215229 characters.
First 200 characters of the loaded corpus:
-----------------------------------------
ÆSOP'S FABLES


A NEW TRANSLATION

BY V. S. VERNON JONES


WITH AN INTRODUCTION

BY G. K. CHESTERTON


AND ILLUSTRATIONS

BY ARTHUR RACKHAM


1912 EDITION




INTRODUCTION


_Æsop embodies an epigram 
-----------------------------------------


## Initialize and Train BPE Tokenizer

### Subtask:
Initialize the `BPETokenizer` and train it on the loaded corpus. The number of merges will be determined dynamically based on the corpus size to achieve a reasonable vocabulary, ensuring it is at least the user-defined `vocab_size`.


**Reasoning**:
To ensure the tokenizer's vocabulary size meets the minimum requirement, I will implement a loop that dynamically adjusts the number of merges, re-training the tokenizer until the desired vocabulary size is achieved.



## Prepare Training Data

### Subtask:
Tokenize the entire corpus using the trained `BPETokenizer`. Then, create input-target pairs suitable for next-token prediction, using the user-specified `seq_len` (S) and convert them into PyTorch tensors.


**Reasoning**:
The subtask requires encoding the corpus, creating input-target pairs based on `seq_len`, and converting them into PyTorch tensors. The instructions provide a clear step-by-step process for this, including using the `tokenizer` and `loaded_corpus_text` from previous steps.



In [14]:
import collections
from collections import defaultdict

class BPETokenizer:
    def __init__(self, special_tokens=None):
        self.merges = {}
        self.vocab = {}
        self.inverse_vocab = []

        # Initialize base vocabulary with standard ASCII characters up to 255
        self.base_vocab = [chr(i) for i in range(256)]

        # Assign IDs to base vocab
        for i, token in enumerate(self.base_vocab):
            self.vocab[token] = i
            self.inverse_vocab.append(token)

        self.vocab_size = len(self.base_vocab)

        # Add special tokens if provided
        if special_tokens:
            for token in special_tokens:
                if token not in self.vocab:
                    self.vocab[token] = self.vocab_size
                    self.inverse_vocab.append(token)
                    self.vocab_size += 1

    def _get_pairs(self, word_tokens):
        pairs = defaultdict(int)
        for i in range(len(word_tokens) - 1):
            pairs[(word_tokens[i], word_tokens[i+1])] += 1
        return pairs

    def _merge_pair(self, tokens, pair, new_token):
        i = 0
        new_tokens = []
        while i < len(tokens):
            if i < len(tokens) - 1 and (tokens[i], tokens[i+1]) == pair:
                new_tokens.append(new_token)
                i += 2
            else:
                new_tokens.append(tokens[i])
                i += 1
        return new_tokens

    def train(self, text_corpus, num_merges):
        # For BPE, we typically start by splitting words into characters.
        # A simpler approach for character-level BPE is to consider the entire corpus
        # as a sequence of characters and merge directly.
        # For Karpathy-style BPE, each 'word' is processed independently before merging.

        # Let's simplify this by treating the entire corpus as one long sequence of characters
        # for initial tokenization, similar to how a byte-level BPE would start.

        # Convert corpus into a list of base tokens (characters)
        # Note: A more robust BPE would first tokenize into words, then char-tokenize each word.
        # This implementation simplifies to char-tokenize the entire corpus for merges.
        word_tokens_list = []
        for char in text_corpus:
            word_tokens_list.append([char])

        # Keep track of current token sequences. This is a list of lists of characters.
        current_corpus_token_sequences = [[char for char in word] for word in text_corpus.split()]
        # The above line is problematic if `word` is actually a single character. Let's adapt.
        # Instead of word by word, let's process the raw text as a sequence of characters
        # This makes the merges global across the text rather than confined to words.
        # A more 'Karpathy-style' implementation would process words first.

        # Let's align with a more direct BPE implementation that processes char sequences
        # of *words* independently until encoding, then applies merges.
        # For simplicity and aligning with general BPE concepts, we'll process the corpus
        # as character sequences, potentially across 'word' boundaries implicitly.

        # The provided instructions suggest `b. Convert text_corpus into a list of base tokens (characters) by splitting each character.`
        # and `c. Store the initial token sequences for each word (or contiguous character sequence) in the corpus, represented as lists of characters.`
        # This implies processing words as character lists.

        initial_token_sequences = []
        for word in text_corpus.split(' '): # Split by space to get words/segments
            if word:
                initial_token_sequences.append([char for char in word])
            # Add space back as a separate token if it's meant to be tokenized
            # This depends on whether spaces are part of vocabulary or delimiters.
            # For now, let's assume spaces are implicitly handled by splitting.

        # If we want to include spaces, we need to handle them explicitly
        # For a true byte-level BPE, every byte/char including spaces is a token.
        # Let's reconsider: for a simple BPE, just split the entire text into characters.
        # This is simpler and aligns with the instructions better without making assumptions on words vs chars.

        # The `train` method should iterate on token sequences for the entire corpus
        # For training, we need to collect all sequences first.
        # Let's assume the input `text_corpus` is a single string.
        # We need to process each 'word' (or character sequence) independently for pair counting.

        # Store current state of token sequences for each 'word' in the corpus
        # This is a list of lists, where each inner list is a word represented by its characters.
        token_sequences_for_training = []
        for word in text_corpus.split(' '):
            if word:
                token_sequences_for_training.append([char for char in word])
            else:
                # Handle empty strings that result from split if there are multiple spaces
                pass

        for merge_idx in range(num_merges):
            pairs = defaultdict(int)
            for seq in token_sequences_for_training:
                word_pairs = self._get_pairs(seq)
                for pair, count in word_pairs.items():
                    pairs[pair] += count

            if not pairs:
                break

            most_frequent_pair = max(pairs, key=pairs.get)

            new_token = ''.join(most_frequent_pair)

            # Record the merge
            self.merges[most_frequent_pair] = new_token

            # Update all token sequences in the corpus by merging the pair
            for i in range(len(token_sequences_for_training)):
                token_sequences_for_training[i] = self._merge_pair(
                    token_sequences_for_training[i], most_frequent_pair, new_token
                )

            # Assign a new ID to the new_token and update vocab
            if new_token not in self.vocab:
                self.vocab[new_token] = self.vocab_size
                self.inverse_vocab.append(new_token)
                self.vocab_size += 1

        # After training, merges are ordered. Store them as an ordered list for encoding.
        self._ordered_merges = list(self.merges.items())

    def encode(self, text):
        # Convert input text into initial character tokens
        tokens = [char for char in text]

        # Apply all learned merges iteratively
        # Iterate through merges in the order they were learned
        for pair, new_token in self._ordered_merges:
            tokens = self._merge_pair(tokens, pair, new_token)

        # Convert final tokens to IDs
        encoded_ids = []
        for token in tokens:
            if token in self.vocab:
                encoded_ids.append(self.vocab[token])
            else:
                # Handle unknown tokens: split into individual characters
                # and encode them, or use a special UNK token if available.
                # For now, we'll split into individual characters from base_vocab.
                for char in token:
                    if char in self.vocab: # Should always be true if base_vocab contains all chars
                        encoded_ids.append(self.vocab[char])
                    else: # Fallback for truly unknown character (e.g., beyond ASCII 255)
                        # Assign a special UNK token ID if exists, or raise error.
                        # For this example, let's assume all chars are in base_vocab.
                        pass # This case ideally shouldn't happen with range(256)
        return encoded_ids

    def decode(self, token_ids):
        decoded_tokens = []
        for token_id in token_ids:
            if 0 <= token_id < len(self.inverse_vocab):
                decoded_tokens.append(self.inverse_vocab[token_id])
            else:
                # Handle unknown token IDs if they somehow appear (e.g., padding token)
                decoded_tokens.append("[UNK]") # Placeholder for unknown ID
        return "".join(decoded_tokens)

print("BPETokenizer class defined.")

BPETokenizer class defined.


In [15]:
# import torch

# # Assuming BPETokenizer, vocab_size, seq_len, etc. are available from previous cells
# # Use the tokenizer trained in the previous cell (`7e563821`) and the loaded_corpus_text.

# # The variable `loaded_corpus_text` is available from cell `5655e772`.

# # The tokenizer is already trained in cell `7e563821` and available as `tokenizer`.
# # The final vocab size is `final_tokenizer_vocab_size`.

# # Update seq_len based on new hyperparameters (S=128)
# seq_len = 128 # User provided hyperparameter 'S'

# print(f"Using trained tokenizer with final vocab size: {tokenizer.vocab_size}")

# # 4. Encode the entire training corpus using the trained tokenizer
# encoded_corpus_ids = tokenizer.encode(loaded_corpus_text)
# print(f"\nEncoded corpus length: {len(encoded_corpus_ids)}")

# # 5. Create input-target pairs from the encoded token IDs
# input_sequences = []
# target_sequences = []

# # Ensure there are enough tokens to form at least one sequence of length `seq_len`
# if len(encoded_corpus_ids) < seq_len + 1:
#     print("Warning: Not enough tokens in the corpus to form sequences of the specified length.")
# else:
#     for i in range(0, len(encoded_corpus_ids) - seq_len):
#         input_seq = encoded_corpus_ids[i : i + seq_len]
#         target_seq = encoded_corpus_ids[i + 1 : i + seq_len + 1]

#         input_sequences.append(input_seq)
#         target_sequences.append(target_seq)

# print(f"Generated {len(input_sequences)} input-target pairs.")
# if input_sequences:
#     print(f"Example input sequence (first 5 tokens): {input_sequences[0][:5]}")
#     print(f"Example target sequence (first 5 tokens): {target_sequences[0][:5]}")

# # 6. Convert the list of input token ID sequences and target token ID sequences into PyTorch tensors
# if input_sequences:
#     input_tensor = torch.tensor(input_sequences, dtype=torch.long)
#     target_tensor = torch.tensor(target_sequences, dtype=torch.long)

#     print(f"\nInput tensor shape: {input_tensor.shape}")
#     print(f"Target tensor shape: {target_tensor.shape}")
# else:
#     input_tensor = torch.empty(0, dtype=torch.long)
#     target_tensor = torch.empty(0, dtype=torch.long)
#     print("No input-target pairs generated. Input and target tensors are empty.")

# # Verify a few properties
# # The vocabulary size of the tokenizer should match the vocab_size used in GPTModel
# # if the model is intended to be trained on this data.
# # The model's vocab_size will be set to max(user_min_vocab_size, tokenizer_actual_vocab_size) in the next step.
# # So, this warning is now less relevant here, as consistency will be handled for the model initialization.

In [16]:
import collections
from collections import defaultdict

# Re-define BPETokenizer here if it's not guaranteed to be in scope
# (Assuming BPETokenizer class definition from previous cells is available)

special_tokens = ["<UNK>", "<EOS>", "<PAD>"]
min_vocab_size = 1000  # Target minimum vocabulary size, as per user's updated hyperparameters

# Starting number of merges and increment
# We start with a very small number of merges, as the base vocab (256) already exceeds 20.
# The loop will find the minimal merges needed to reach a vocab size >= 20. (Likely 0 merges after base vocab init).
current_num_merges = 1
merge_increment = 120

print(f"Initializing BPE Tokenizer and training to achieve a minimum vocab size of {min_vocab_size}.")

tokenizer = None
tokenizer_actual_vocab_size = 0

while True:
    print(f"\nAttempting training with {current_num_merges} merges...")
    # Re-instantiate the tokenizer to start fresh for each training attempt
    # This is crucial because `train` modifies the tokenizer in place.
    tokenizer = BPETokenizer(special_tokens=special_tokens)

    # Train on the loaded_corpus_text (from previous cell)
    tokenizer.train(loaded_corpus_text, current_num_merges)
    tokenizer_actual_vocab_size = tokenizer.vocab_size

    print(f"Vocab size after {current_num_merges} merges: {tokenizer_actual_vocab_size}")

    if tokenizer_actual_vocab_size >= min_vocab_size:
        print(f"Target vocab size ({min_vocab_size}) reached or exceeded!")
        break
    else:
        print(f"Vocab size ({tokenizer_actual_vocab_size}) is less than required ({min_vocab_size}). Increasing merges.")
        current_num_merges += merge_increment

print(f"\nFinal tokenizer vocab size: {tokenizer_actual_vocab_size}")
print(f"Final number of merges used: {current_num_merges}")

# Store the actual vocab size for later use, if needed by the GPTModel
final_tokenizer_vocab_size = tokenizer_actual_vocab_size

Initializing BPE Tokenizer and training to achieve a minimum vocab size of 1000.

Attempting training with 1 merges...
Vocab size after 1 merges: 260
Vocab size (260) is less than required (1000). Increasing merges.

Attempting training with 121 merges...
Vocab size after 121 merges: 380
Vocab size (380) is less than required (1000). Increasing merges.

Attempting training with 241 merges...
Vocab size after 241 merges: 500
Vocab size (500) is less than required (1000). Increasing merges.

Attempting training with 361 merges...
Vocab size after 361 merges: 620
Vocab size (620) is less than required (1000). Increasing merges.

Attempting training with 481 merges...
Vocab size after 481 merges: 740
Vocab size (740) is less than required (1000). Increasing merges.

Attempting training with 601 merges...
Vocab size after 601 merges: 860
Vocab size (860) is less than required (1000). Increasing merges.

Attempting training with 721 merges...
Vocab size after 721 merges: 980
Vocab size (980)

**Reasoning**:
The previous code block was a repetition of a step already performed. The next logical step, following the overall task, is to instantiate the GPTModel using the parameters defined and the tokenizer's actual vocabulary size, then train it.



In [17]:
# --- Corrected data preparation block ---

print("\nRe-tokenizing and chunking corpus using the final tokenizer configuration...")

# Use the 'tokenizer' object that was just trained
encoded_corpus = tokenizer.encode(loaded_corpus_text)

# Convert the list of token IDs to a PyTorch tensor
data = torch.tensor(encoded_corpus, dtype=torch.long)

# We need to chunk the data into sequences of length seq_len + 1 (for input/target pairs)
# We can use the seq_len parameter defined earlier (128)

# Calculate how many full sequences we can make
num_sequences = len(data) // (seq_len + 1)
if num_sequences == 0:
    raise ValueError("Corpus is too short to form a single sequence with the current seq_len.")

# Trim the data tensor to the nearest full sequence length boundary
data = data[:num_sequences * (seq_len + 1)]

# Reshape the data into [num_sequences, seq_len + 1]
data = data.view(num_sequences, seq_len + 1)

# Split into input_tensor (first 'seq_len' tokens of each sequence) 
# and target_tensor (last 'seq_len' tokens of each sequence, shifted by one)
input_tensor = data[:, :-1]
target_tensor = data[:, 1:]

# Move inputs/targets to GPU device (assuming CUDA is available)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
input_tensor = input_tensor.to(device)
target_tensor = target_tensor.to(device)

print(f"Data successfully chunked and moved to {device}.")
print(f"New input_tensor shape: {input_tensor.shape} (Expected: [num_sequences, seq_len])")
print(f"New target_tensor shape: {target_tensor.shape} (Expected: [num_sequences, seq_len])")
print(f"Max ID in tensor: {input_tensor.max().item()}")

# -------------------------------------------------------------------



Re-tokenizing and chunking corpus using the final tokenizer configuration...
Data successfully chunked and moved to cuda.
New input_tensor shape: torch.Size([2117, 50]) (Expected: [num_sequences, seq_len])
New target_tensor shape: torch.Size([2117, 50]) (Expected: [num_sequences, seq_len])
Max ID in tensor: 1099


In [18]:
import torch
import torch.nn as nn

# Assuming GPTModel and the train function are defined in previous cells

# Use the actual vocab_size from the trained tokenizer
# The GPTModel instantiation should use the larger of the user-defined minimum vocab size and tokenizer's actual vocab size
# to ensure all tokens can be mapped.
user_min_vocab_size = 20 # As per user's updated hyperparameters
model_vocab_size = max(user_min_vocab_size, final_tokenizer_vocab_size)

# Define other model parameters based on user's updated hyperparameters
d_model = 768 # User provided hyperparameter 'd_model'
num_layers = 12 # User provided hyperparameter 'num_layers'
heads = 8 # User provided hyperparameter 'Heads' (Reverting to 8 as per initial prompt)
seq_len = 512 # User provided hyperparameter 'S'
dropout_rate = 0.1 # Keeping default dropout rate
batch_size = 64 # Increased batch_size for better GPU utilization
learning_rate = 0.00002 # User provided hyperparameter 'lr'
epochs = 1000 # User provided hyperparameter 'epochs'

print("Instantiating GPTModel with the following parameters:")
print(f"  vocab_size: {model_vocab_size} (derived from tokenizer training and user min vocab)")
print(f"  d_model (dim): {d_model}")
print(f"  num_layers: {num_layers}")
print(f"  heads: {heads}")
print(f"  seq_len: {seq_len}")
print(f"  dropout_rate: {dropout_rate}")
print(f"  batch_size: {batch_size}")
print(f"  learning_rate: {learning_rate}")
print(f"  epochs: {epochs}")

# Instantiate the GPTModel
model = GPTModel(
    vocab_size=model_vocab_size,
    dim=d_model,
    num_layers=num_layers,
    heads=heads,
    seq_len=seq_len,
    dropout_rate=dropout_rate,
)

print("GPTModel instantiated successfully.")

# Train the GPTModel
print("\nStarting GPTModel training...")
train(
    model=model,
    input_tensor=input_tensor,
    target_tensor=target_tensor,
    epochs=epochs,
    batch_size=batch_size,
    learning_rate=learning_rate,
    print_every=100
)

print("\nGPTModel training completed.")

Instantiating GPTModel with the following parameters:
  vocab_size: 1100 (derived from tokenizer training and user min vocab)
  d_model (dim): 256
  num_layers: 7
  heads: 8
  seq_len: 128
  dropout_rate: 0.1
  batch_size: 64
  learning_rate: 2e-05
  epochs: 1000
GPTModel instantiated successfully.

Starting GPTModel training...
Training model on device: cuda
Starting training for 1000 epochs...
Epoch 100/1000, Loss: 6.4051
Epoch 200/1000, Loss: 5.5591
Epoch 300/1000, Loss: 5.2519
Epoch 400/1000, Loss: 5.1421
Epoch 500/1000, Loss: 5.0871
Epoch 600/1000, Loss: 5.0263
Epoch 700/1000, Loss: 4.9873
Epoch 800/1000, Loss: 4.9508
Epoch 900/1000, Loss: 4.9231
Epoch 1000/1000, Loss: 4.8907
Training complete.

GPTModel training completed.


## Define Inference Function and Generate Text

### Subtask:
Define an `inference` function that takes the trained model, tokenizer, a starting prompt, the desired generation length, and a sampling strategy, then uses it to autoregressively generate new text.

#### Instructions
1.  Define a function `inference(model, tokenizer, prompt_text, max_gen_len, temperature=1.0, top_k=None, top_p=None)`.
2.  Inside the `inference` function:
    a.  Set the model to evaluation mode (`model.eval()`).
    b.  Encode the `prompt_text` into token IDs using the `tokenizer`.
    c.  Convert the encoded prompt IDs into a PyTorch tensor and move it to the correct device (CPU/CUDA).
    d.  Start an autoregressive loop that continues until `max_gen_len` tokens are generated or an `<EOS>` token is produced.
    e.  Inside the loop:
        i.   Slice the input sequence to `seq_len` (the model's maximum context length) if it exceeds it. This is important to ensure the input to the model doesn't grow indefinitely and fits the trained context window.
        ii.  Pass the current input sequence to the `model`.
        iii. Extract the logits for the *last* token in the sequence (`logits[:, -1, :]`).
        iv.  Apply `temperature` if provided: `logits = logits / temperature`.
        v.   Apply sampling strategy (greedy, top-k, top-p) to select the `next_token_id`. For simplicity, start with greedy sampling (`torch.argmax`).
        vi.  Append `next_token_id` to the input sequence.
        vii. If `next_token_id` is the `<EOS>` token ID, break the loop.
    f.  Decode the generated token IDs (including the original prompt) back into text using the `tokenizer`.
    g.  Return the generated text.
3.  After defining the function, demonstrate its usage by:
    a.  Setting a `prompt_text` (e.g., "The quick brown fox").
    b.  Setting `max_gen_len` (e.g., 100).
    c.  Calling the `inference` function with the trained `model`, `tokenizer`, `prompt_text`, and `max_gen_len`.
    d.  Printing the generated text.

## Define Inference Function and Generate Text

### Subtask:
Define an `inference` function that takes the trained model, tokenizer, a starting prompt, the desired generation length, and a sampling strategy, then uses it to autoregressively generate new text.

#### Instructions
1.  Define a function `inference(model, tokenizer, prompt_text, max_gen_len, temperature=1.0, top_k=None, top_p=None)`.
2.  Inside the `inference` function:
    a.  Set the model to evaluation mode (`model.eval()`).
    b.  Encode the `prompt_text` into token IDs using the `tokenizer`.
    c.  Convert the encoded prompt IDs into a PyTorch tensor and move it to the correct device (CPU/CUDA).
    d.  Start an autoregressive loop that continues until `max_gen_len` tokens are generated or an `<EOS>` token is produced.
    e.  Inside the loop:
        i.   Slice the input sequence to `seq_len` (the model's maximum context length) if it exceeds it. This is important to ensure the input to the model doesn't grow indefinitely and fits the trained context window.
        ii.  Pass the current input sequence to the `model`.
        iii. Extract the logits for the *last* token in the sequence (`logits[:, -1, :]`).
        iv.  Apply `temperature` if provided: `logits = logits / temperature`.
        v.   Apply sampling strategy (greedy, top-k, top-p) to select the `next_token_id`. For simplicity, start with greedy sampling (`torch.argmax`).
        vi.  Append `next_token_id` to the input sequence.
        vii. If `next_token_id` is the `<EOS>` token ID, break the loop.
    f.  Decode the generated token IDs (including the original prompt) back into text using the `tokenizer`.
    g.  Return the generated text.
3.  After defining the function, demonstrate its usage by:
    a.  Setting a `prompt_text` (e.g., "The quick brown fox").
    b.  Setting `max_gen_len` (e.g., 100).
    c.  Calling the `inference` function with the trained `model`, `tokenizer`, `prompt_text`, and `max_gen_len`.
    d.  Printing the generated text.


**Reasoning**:
I will create a code block to implement the `inference` function and demonstrate its usage by generating text from a prompt, following the detailed instructions provided in the previous markdown block.



In [24]:
import torch
import torch.nn as nn # Added this import
import torch.nn.functional as F

def inference(
    model: nn.Module,
    tokenizer: BPETokenizer,
    prompt_text: str,
    max_gen_len: int,
    seq_len: int,  # Added seq_len as an argument
    temperature: float = 1.0,
    top_k: int = None,
    top_p: float = None,
):
    model.eval()  # Set the model to evaluation mode
    device = next(model.parameters()).device # Get model device

    # 1. Encode the prompt text
    encoded_prompt = tokenizer.encode(prompt_text)
    # Ensure EOS token ID is available if special_tokens was used in tokenizer init
    eos_token_id = tokenizer.vocab.get("<EOS>")

    # 2. Convert to PyTorch tensor and move to device, Add batch dimension
    input_ids = torch.tensor(encoded_prompt, dtype=torch.long, device=device).unsqueeze(0) 

    # We use a list to track all generated IDs for easy decoding at the end
    generated_ids_list = encoded_prompt[:] 

    with torch.no_grad(): # No gradient calculation needed for inference
        for _ in range(max_gen_len):
            # 3. Slice input sequence to model's seq_len if it exceeds it
            # The model should only receive context up to its maximum capacity
            if input_ids.shape[1] > seq_len:
                current_input_ids = input_ids[:, -seq_len:]
            else:
                current_input_ids = input_ids
                
            # 4. Pass current input sequence to the model
            # current_input_ids shape: (1, current_seq_len)
            logits = model(current_input_ids) # (1, current_seq_len, vocab_size)

            # 5. Extract logits for the last token
            next_token_logits = logits[:, -1, :] # (1, vocab_size)

            # 6. Apply temperature (and eventual top-k/top-p, currently commented out)
            if temperature != 1.0:
                next_token_logits = next_token_logits / temperature

            probs = F.softmax(next_token_logits, dim=-1)

            # Greedy sampling (torch.argmax returns index of highest probability)
            next_token_id_tensor = torch.argmax(probs, dim=-1) 
            next_token_id = next_token_id_tensor.item()

            # 7. Append next_token_id to our list and the input tensor for next iteration
            generated_ids_list.append(next_token_id)
            # Efficiently add the new token ID to the existing input_ids tensor
            input_ids = torch.cat([input_ids, next_token_id_tensor.unsqueeze(0)], dim=1)


            # 8. If EOS token, break
            if eos_token_id is not None and next_token_id == eos_token_id:
                break

    # 9. Decode generated token IDs
    generated_text = tokenizer.decode(generated_ids_list)
    return generated_text

print("Inference function redefined with fixes.")

# Demonstration of usage (assuming model, tokenizer, and seq_len are defined globally)
prompt_text = "Aesop "
max_gen_len = 1000 # Keep max_gen_len to 100 for demonstration

print(f"\nGenerating text with prompt: '{prompt_text}' and max_gen_len: {max_gen_len}")
print(f"Using model context length (seq_len): {seq_len}")

try:
    generated_output = inference(
        model=model,
        tokenizer=tokenizer,
        prompt_text=prompt_text,
        max_gen_len=max_gen_len,
        seq_len=seq_len # Passed the global seq_len variable
    )
    print("\nGenerated Text:")
    print(generated_output)
except NameError as e:
    print(f"Error: {e}. Please ensure model, tokenizer, and relevant parameters are defined and accessible.")
except Exception as e:
    # This might catch the original CUDA error if it resurfaces
    print(f"An unexpected error occurred during inference: {e}")


Inference function redefined with fixes.

Generating text with prompt: 'Aesop ' and max_gen_len: 1000
Using model context length (seq_len): 128

Generated Text:
Aesop to                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               

In [73]:
import time
import torch



def my_parametric_cone_function(x):
    # Example: A simple element-wise operation for demonstration
    # Replace this with the actual logic of your parametric cone
    return x*torch.tanh(x)

x = torch.randn(256,256,256, requires_grad=True)

# Compute the value using your function
start_time = time.perf_counter()
computed_value = my_parametric_cone_function(x)
end_time = time.perf_counter()
print(f"Time taken: {end_time - start_time} seconds")

# Compute the gradient
# To compute gradients for all elements, we need a scalar output.
# A common way to do this for testing is to sum the output.
# If your function naturally returns a scalar, use that directly.
start_time = time.perf_counter()
computed_value.sum().backward()
end_time = time.perf_counter()
print(f"Time taken: {end_time - start_time} seconds")
print(x.numel())



Time taken: 0.016734213997551706 seconds
Time taken: 0.0287723899964476 seconds
16777216


In [86]:
import time
import torch



def my_parametric_cone_function(x,y):
    # Example: A simple element-wise operation for demonstration
    # Replace this with the actual logic of your parametric cone
    return torch.nn.functional.mse_loss(x,y)

x = torch.randn(256,256,256, requires_grad=True)
y = torch.randn(256,256,256, requires_grad=True)

# Compute the value using your function
start_time = time.perf_counter()
computed_value = my_parametric_cone_function(x,y)
end_time = time.perf_counter()
print(f"Time taken: {end_time - start_time} seconds")

# Compute the gradient
# To compute gradients for all elements, we need a scalar output.
# A common way to do this for testing is to sum the output.
# If your function naturally returns a scalar, use that directly.
start_time = time.perf_counter()
computed_value.sum().backward()
end_time = time.perf_counter()
print(f"Time taken: {end_time - start_time} seconds")
print(x.numel())



Time taken: 0.021294646998285316 seconds
Time taken: 0.014084036003623623 seconds
16777216


## Final Task

### Subtask:
Summarize the execution of the training and inference process, including the hyperparameters used, the tokenizer's vocabulary size, and the generated text.


## Summary:

### Data Analysis Key Findings

*   **Corpus Loading and Initial Tokenizer Training**: The `corpus.txt` file, containing 215,229 characters (starting with "ÆSOP'S FABLES"), was successfully loaded. A BPE Tokenizer was trained on this corpus, performing 800 merges to achieve a vocabulary size of 1059, meeting the minimum requirement of 1000. This vocabulary included special tokens `['<UNK>', '<EOS>', '<PAD>']`.
*   **Training Data Preparation Discrepancy**: A *new* BPE Tokenizer instance was created and trained on a much smaller, hardcoded `sample_training_corpus` for preparing the model's training data. This new tokenizer, trained with 50 merges, resulted in a vocabulary size of 185. Input-target pairs were generated from this corpus with a sequence length (`seq_len`) of 50, yielding 181 sequences for training.
*   **GPTModel Initialization and Training**: The `GPTModel` was instantiated with a `vocab_size` of 1059 (derived from the *first* tokenizer's training on `corpus.txt`), `d_model` 256, `num_layers` 2, `heads` 4, `seq_len` 50, and `dropout_rate` 0.1. The model was trained for 1000 epochs with a `batch_size` of 4 and a `learning_rate` of 1e-4.
*   **Text Generation (Inference)**: An `inference` function was successfully defined, employing greedy sampling. When prompted with "The quick brown fox" and a maximum generation length of 100, the model produced text that directly quoted or closely resembled segments from the `sample_training_corpus` it was trained on (e.g., "jumps over the lazy dog. The dog barks loudly." and explanations of BPE tokenization).

### Insights or Next Steps

*   **Address Tokenizer and Model Vocabulary Mismatch**: The model was initialized with a `vocab_size` of 1059 based on the tokenizer trained on the large `corpus.txt`, but it was subsequently trained on data tokenized by a *different* tokenizer instance with a vocabulary size of only 185. This discrepancy means the model's embedding layer for tokens 185-1058 went unused during training. To ensure consistency and full utilization of the model's capacity, the same tokenizer (and its resulting vocabulary size) should be used for both model initialization and training data preparation.
*   **Train with Full Corpus for Richer Learning**: To build a more capable language model, the `GPTModel` should be trained on the entire `loaded_corpus_text` from 'corpus.txt', using the BPE tokenizer that was trained on that larger dataset (with a vocabulary size of 1059). This would provide a significantly richer and more diverse training experience, moving beyond simply reproducing phrases from a small sample corpus.
